In [8]:
from pathlib import Path

# Resolve project root regardless of whether notebook runs from repo root or /notebooks.
if (Path.cwd() / 'cs505_data').exists():
  PROJECT_ROOT = Path.cwd()
else:
  PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / 'cs505_data'

fr_path = DATA_DIR / 'europarl-v7.fr-en.fr'
en_path = DATA_DIR / 'europarl-v7.fr-en.en'

with open(fr_path, encoding='utf-8') as f:
  fr_lines = f.readlines()
with open(en_path, encoding='utf-8') as f:
  en_lines = f.readlines()

print(f"Total sentence pairs: {len(fr_lines):,}")
print(f"FR: {fr_lines[0].strip()}")
print(f"EN: {en_lines[0].strip()}")

Total sentence pairs: 2,007,723
FR: Reprise de la session
EN: Resumption of the session


In [9]:
from collections import Counter, defaultdict
import math
import re
import sacrebleu

def tokenize(sentence):
  # Must match fast_align tokenization: whitespace + lowercase.
  return sentence.lower().strip().split()

def detokenize(tokens):
  return ' '.join(tokens)

# Training budget (max sentence pairs used to build phrase/LM tables).
TRAIN_SIZE = 50000

# Koehn et al. 2003 §4: hold out 1,755 test sentences with source length 5–15
# (they did this on German length; we use French source length the same way).
EUROPARL_TEST_SIZE = 1755
TEST_SRC_MIN_LEN = 5
TEST_SRC_MAX_LEN = 15

# Sentinel used by the lexical translation tables for NULL alignments.
NULL_TOKEN = '<NULL>'

print(f"Train cap: {TRAIN_SIZE:,} | Europarl held-out test: {EUROPARL_TEST_SIZE} "
      f"(FR len {TEST_SRC_MIN_LEN}-{TEST_SRC_MAX_LEN})")

Train cap: 50,000 | Europarl held-out test: 1755 (FR len 5-15)


In [10]:
# Symmetrized word alignments (Pharaoh i-j format) from fast_align / GIZA++.
# File: cs505_data/train.fr-en.gdfa  (one line per pair like "0-0 1-2 2-1")

def parse_pharaoh_alignment(line):
  links = set()
  for pair in line.strip().split():
    if '-' not in pair:
      continue
    i_str, j_str = pair.split('-', 1)
    if i_str.isdigit() and j_str.isdigit():
      links.add((int(i_str), int(j_str)))
  return links

def load_alignments(path, max_sentences=None):
  alignments = []
  with open(path, encoding='utf-8') as f:
    for idx, line in enumerate(f):
      if max_sentences is not None and idx >= max_sentences:
        break
      alignments.append(parse_pharaoh_alignment(line))
  return alignments

def get_phrases_from_alignment(fr_sent, en_sent, links, max_len=3):
  """Och et al. / Koehn et al. extraction: every phrase pair consistent with
  the word alignment. Returns (fr_phrase, en_phrase, local_links) triples
  where local_links is the list of (i_local, j_local) alignment points inside
  the phrase span — required to compute lexical weights p_w(e|f,a) later.
  """
  lf, le = len(fr_sent), len(en_sent)
  if lf == 0 or le == 0:
    return []

  aligned_by_e = defaultdict(set)
  aligned_by_f = defaultdict(set)
  for i, j in links:
    if i < lf and j < le:
      aligned_by_e[j].add(i)
      aligned_by_f[i].add(j)

  phrases = []
  for i1 in range(lf):
    js_set = set()
    for i2 in range(i1, min(i1 + max_len, lf)):
      js_set.update(aligned_by_f.get(i2, set()))
      if not js_set:
        continue

      j1, j2 = min(js_set), max(js_set)
      if (j2 - j1 + 1) > max_len:
        continue

      # Consistency: no EN word in (j1..j2) aligns outside (i1..i2).
      consistent = True
      for j in range(j1, j2 + 1):
        for i_aligned in aligned_by_e.get(j, set()):
          if i_aligned < i1 or i_aligned > i2:
            consistent = False
            break
        if not consistent:
          break
      if not consistent:
        continue

      local_links = tuple(
          (i - i1, j - j1)
          for i, j in links
          if i1 <= i <= i2 and j1 <= j <= j2
      )
      phrases.append((
          tuple(fr_sent[i1:i2 + 1]),
          tuple(en_sent[j1:j2 + 1]),
          local_links,
      ))

  return phrases


In [11]:
# Koehn et al. 2003 §4.3: "length 3 is enough" (longer phrases do not help).
PHRASE_MAX_LEN = 3
# Paper does not filter by count; we keep singletons (we only have 50k pairs).
MIN_PHRASE_COUNT = 1
# Keep more candidates so the decoder has room to pick a good one.
TOP_K_TRANSLATIONS = 20

PARALLEL_PATH = DATA_DIR / 'train.fr-en.clean.parallel'
ALIGN_PATH = DATA_DIR / 'train.fr-en.gdfa'

def load_parallel_corpus(path, max_lines=None):
  """Load bitext — must stay line-aligned with train.fr-en.gdfa (no skipped rows)."""
  fr_tok, en_tok, fr_raw, en_raw = [], [], [], []
  with open(path, encoding='utf-8') as f:
    for idx, line in enumerate(f):
      if max_lines is not None and idx >= max_lines:
        break
      if '|||' not in line:
        raise ValueError(f'{path}:{idx+1}: expected ||| delimiter')
      fr_r, en_r = line.split('|||', 1)
      fr_r, en_r = fr_r.strip(), en_r.strip()
      fto, eto = tokenize(fr_r), tokenize(en_r)
      if not fto or not eto:
        raise ValueError(f'{path}:{idx+1}: empty tokens after tokenize')
      fr_tok.append(fto)
      en_tok.append(eto)
      fr_raw.append(fr_r)
      en_raw.append(en_r)
  return fr_tok, en_tok, fr_raw, en_raw

# --- Lexical translation tables with NULL (paper §4.4) -----------------------
# p_w(e | f): for each alignment point (i,j), credit count(f_i, e_j).
# Every unaligned English word is aligned to a virtual NULL source word,
# so the decoder can score unaligned target words later.

def build_lex_tables(fr_corpus, en_corpus, alignments):
  count_e_given_f = defaultdict(Counter)  # indexed by source word (or NULL)
  count_f_given_e = defaultdict(Counter)  # indexed by target word (or NULL)

  for fr_sent, en_sent, links in zip(fr_corpus, en_corpus, alignments):
    lf, le = len(fr_sent), len(en_sent)
    aligned_f, aligned_e = set(), set()
    for i, j in links:
      if i < lf and j < le:
        count_e_given_f[fr_sent[i]][en_sent[j]] += 1
        count_f_given_e[en_sent[j]][fr_sent[i]] += 1
        aligned_f.add(i)
        aligned_e.add(j)
    for j in range(le):
      if j not in aligned_e:
        count_e_given_f[NULL_TOKEN][en_sent[j]] += 1
    for i in range(lf):
      if i not in aligned_f:
        count_f_given_e[NULL_TOKEN][fr_sent[i]] += 1

  pw_e_given_f = {}
  for f, ctr in count_e_given_f.items():
    total = sum(ctr.values())
    pw_e_given_f[f] = {e: c / total for e, c in ctr.items()}

  pw_f_given_e = {}
  for e, ctr in count_f_given_e.items():
    total = sum(ctr.values())
    pw_f_given_e[e] = {f: c / total for f, c in ctr.items()}

  return pw_e_given_f, pw_f_given_e

def build_fallback_lexicon(pw_e_given_f):
  """Single most-probable English translation per French word (ignoring NULL)."""
  fallback = {}
  for f, dist in pw_e_given_f.items():
    if f == NULL_TOKEN:
      continue
    ranked = sorted(dist.items(), key=lambda x: x[1], reverse=True)
    for e, _ in ranked:
      if e != NULL_TOKEN:
        fallback[f] = e
        break
  return fallback

# --- Lexical weighting (paper §4.4, Figure 3) --------------------------------
# p_w(e_bar | f_bar, a) = prod_{j=1..n} (1/|{i:(i,j)∈a}|) * sum p_w(e_j|f_i)
# For unaligned e_j, use p_w(e_j | NULL).

def lex_weight_e_given_f(fr_p, en_p, local_links, pw_e_given_f):
  aligned_to_e = defaultdict(list)
  for i_loc, j_loc in local_links:
    aligned_to_e[j_loc].append(i_loc)
  prod = 1.0
  for j_loc, e_word in enumerate(en_p):
    srcs = aligned_to_e.get(j_loc)
    if srcs:
      s = sum(pw_e_given_f.get(fr_p[i_loc], {}).get(e_word, 1e-9) for i_loc in srcs)
      prod *= s / len(srcs)
    else:
      prod *= pw_e_given_f.get(NULL_TOKEN, {}).get(e_word, 1e-9)
  return prod

def lex_weight_f_given_e(fr_p, en_p, local_links, pw_f_given_e):
  aligned_to_f = defaultdict(list)
  for i_loc, j_loc in local_links:
    aligned_to_f[i_loc].append(j_loc)
  prod = 1.0
  for i_loc, f_word in enumerate(fr_p):
    tgts = aligned_to_f.get(i_loc)
    if tgts:
      s = sum(pw_f_given_e.get(en_p[j_loc], {}).get(f_word, 1e-9) for j_loc in tgts)
      prod *= s / len(tgts)
    else:
      prod *= pw_f_given_e.get(NULL_TOKEN, {}).get(f_word, 1e-9)
  return prod

# --- Phrase table with 4 Moses-style features -------------------------------
# For each extracted (f_bar, e_bar) pair we store:
#   log phi(e|f), log phi(f|e), log lex(e|f), log lex(f|e)

def build_phrase_table(fr_corpus, en_corpus, alignments,
                       pw_e_given_f, pw_f_given_e,
                       max_phrase_len=3, min_count=1, top_k=20):
  pair_count = Counter()
  fr_count = Counter()
  en_count = Counter()
  best_lex_ef = {}
  best_lex_fe = {}

  for fr_sent, en_sent, links in zip(fr_corpus, en_corpus, alignments):
    for fr_p, en_p, loc in get_phrases_from_alignment(
        fr_sent, en_sent, links, max_len=max_phrase_len):
      key = (fr_p, en_p)
      pair_count[key] += 1
      fr_count[fr_p] += 1
      en_count[en_p] += 1
      lw_ef = lex_weight_e_given_f(fr_p, en_p, loc, pw_e_given_f)
      lw_fe = lex_weight_f_given_e(fr_p, en_p, loc, pw_f_given_e)
      # Paper §4.4: "If there are multiple alignments for a phrase pair,
      # we use the one with the highest lexical weight."
      if key not in best_lex_ef or lw_ef > best_lex_ef[key]:
        best_lex_ef[key] = lw_ef
      if key not in best_lex_fe or lw_fe > best_lex_fe[key]:
        best_lex_fe[key] = lw_fe

  phrase_table = defaultdict(list)
  for (fr_p, en_p), c in pair_count.items():
    if c < min_count:
      continue
    phi_ef = c / fr_count[fr_p]
    phi_fe = c / en_count[en_p]
    phrase_table[fr_p].append({
        'en': en_p,
        'log_phi_ef': math.log(phi_ef),
        'log_phi_fe': math.log(phi_fe),
        'log_lex_ef': math.log(max(best_lex_ef[(fr_p, en_p)], 1e-20)),
        'log_lex_fe': math.log(max(best_lex_fe[(fr_p, en_p)], 1e-20)),
    })

  for fr_p in phrase_table:
    phrase_table[fr_p].sort(key=lambda o: o['log_phi_ef'], reverse=True)
    phrase_table[fr_p] = phrase_table[fr_p][:top_k]

  return dict(phrase_table)

# --- Trigram LM with stupid backoff (paper uses a trigram LM in §2.1) -------
# Stupid backoff is a cheap stand-in for Kneser-Ney that beats add-1 a lot.

def build_trigram_lm(en_corpus):
  unigram = Counter()
  bigram = Counter()
  trigram = Counter()
  bigram_ctx = Counter()   # Σ_c trigram[(a,b,c)]
  unigram_ctx = Counter()  # Σ_b bigram[(a,b)]
  total = 0

  for sent in en_corpus:
    seq = ['<s>', '<s>'] + list(sent) + ['</s>']
    for w in seq:
      unigram[w] += 1
      total += 1
    for i in range(1, len(seq)):
      bigram[(seq[i - 1], seq[i])] += 1
      unigram_ctx[seq[i - 1]] += 1
    for i in range(2, len(seq)):
      trigram[(seq[i - 2], seq[i - 1], seq[i])] += 1
      bigram_ctx[(seq[i - 2], seq[i - 1])] += 1

  V = len(unigram)
  log_alpha = math.log(0.4)          # Brants et al. stupid-backoff constant
  log_floor = math.log(1.0 / (total + V))

  def word_logprob(w, c1, c2):
    tri = trigram.get((c1, c2, w), 0)
    if tri:
      return math.log(tri / bigram_ctx[(c1, c2)])
    bi = bigram.get((c2, w), 0)
    if bi and unigram_ctx.get(c2, 0):
      return log_alpha + math.log(bi / unigram_ctx[c2])
    uni = unigram.get(w, 0)
    if uni:
      return 2 * log_alpha + math.log(uni / total)
    return 2 * log_alpha + log_floor

  return word_logprob

# --- Run everything ---------------------------------------------------------
# Paper-style split (Koehn et al. 2003 §4): reserve 1,755 source sentences
# with length 5–15 as in-domain test; train on up to TRAIN_SIZE other pairs.

if not PARALLEL_PATH.exists():
  raise FileNotFoundError(f"Missing cleaned parallel file: {PARALLEL_PATH}")
if not ALIGN_PATH.exists():
  raise FileNotFoundError(f"Missing alignment file: {ALIGN_PATH}")

all_fr, all_en, all_fr_raw, all_en_raw = load_parallel_corpus(PARALLEL_PATH, max_lines=None)
all_align = load_alignments(ALIGN_PATH, max_sentences=len(all_fr))
n = min(len(all_fr), len(all_en), len(all_align), len(all_fr_raw), len(all_en_raw))
all_fr = all_fr[:n]
all_en = all_en[:n]
all_fr_raw = all_fr_raw[:n]
all_en_raw = all_en_raw[:n]
all_align = all_align[:n]

qual_test_idx = [
    i for i in range(n)
    if TEST_SRC_MIN_LEN <= len(all_fr[i]) <= TEST_SRC_MAX_LEN
]
if len(qual_test_idx) < EUROPARL_TEST_SIZE:
  raise ValueError(
      f"Need {EUROPARL_TEST_SIZE} test sentences with "
      f"{TEST_SRC_MIN_LEN}<=|f|<={TEST_SRC_MAX_LEN} FR tokens; found {len(qual_test_idx)}."
  )

# Last qualifying sentences in corpus order (held-out tail among short sentences).
test_idx_set = set(qual_test_idx[-EUROPARL_TEST_SIZE:])
train_idx = [i for i in range(n) if i not in test_idx_set][:TRAIN_SIZE]

test_order = sorted(test_idx_set)
europarl_test_fr_tok = [all_fr[i] for i in test_order]
europarl_test_en_tok = [all_en[i] for i in test_order]
europarl_test_fr_raw = [all_fr_raw[i] for i in test_order]
europarl_test_en_raw = [all_en_raw[i] for i in test_order]

train_fr = [all_fr[i] for i in train_idx]
train_en = [all_en[i] for i in train_idx]
alignments = [all_align[i] for i in train_idx]

print(f"Corpus lines (parallel == align): {n:,}")
print(f"Europarl held-out test: {len(test_order):,} "
      f"(FR length in [{TEST_SRC_MIN_LEN}, {TEST_SRC_MAX_LEN}] tokens)")
print(f"Training pairs: {len(train_fr):,} (cap {TRAIN_SIZE:,})")

print("Building lexical translation tables (with NULL)...")
pw_e_given_f, pw_f_given_e = build_lex_tables(train_fr, train_en, alignments)
print(f"  p_w(e|f) source vocab: {len(pw_e_given_f):,}")
print(f"  p_w(f|e) target vocab: {len(pw_f_given_e):,}")

print("Building fallback lexicon...")
fallback_lexicon = build_fallback_lexicon(pw_e_given_f)
print(f"  Fallback entries: {len(fallback_lexicon):,}")

print("Building phrase table with lex weights...")
phrase_table = build_phrase_table(
    train_fr, train_en, alignments,
    pw_e_given_f, pw_f_given_e,
    max_phrase_len=PHRASE_MAX_LEN,
    min_count=MIN_PHRASE_COUNT,
    top_k=TOP_K_TRANSLATIONS,
)
print(f"  Phrase table source entries: {len(phrase_table):,}")
print(f"  Total (f_bar, e_bar) pairs: {sum(len(v) for v in phrase_table.values()):,}")

print("Building trigram LM (stupid backoff)...")
lm_word_logprob = build_trigram_lm(train_en)
print("Done.")

Corpus lines (parallel == align): 49,837
Europarl held-out test: 1,755 (FR length in [5, 15] tokens)
Training pairs: 48,082 (cap 50,000)
Building lexical translation tables (with NULL)...
  p_w(e|f) source vocab: 50,038
  p_w(f|e) target vocab: 38,394
Building fallback lexicon...
  Fallback entries: 50,037
Building phrase table with lex weights...
  Phrase table source entries: 524,073
  Total (f_bar, e_bar) pairs: 772,876
Building trigram LM (stupid backoff)...
Done.


In [12]:
# ============================================================================
# Log-linear decoder (Koehn et al. 2003, §2.2)
#   - Stacks indexed by number of source words covered (coverage bitmap).
#   - Distortion limit + exponential distortion penalty.
#   - Pre-computed future cost estimate via DP over translation options.
#   - Hypothesis recombination on (coverage, last_end, last two LM words).
# ============================================================================

# Feature weights (manually tuned; retune on a dev set if you have one).
# Higher weight = the feature dominates the score. All log-probs are negative,
# so weights are positive for "good-to-have" features.
WEIGHTS = {
    'log_phi_ef':     1.0,   # p(e_bar | f_bar)
    'log_phi_fe':     0.5,   # p(f_bar | e_bar) — paper's noisy-channel direction
    'log_lex_ef':     0.5,   # lex(e_bar | f_bar, a)
    'log_lex_fe':     0.5,   # lex(f_bar | e_bar, a)
    'lm':             1.0,   # trigram log-prob
    'word_bonus':     0.5,   # paper's omega: +wb per output word (length reward)
    'phrase_penalty': 0.0,   # penalty per phrase used
    'distortion':    -1.0,   # penalty per unit of distortion |a_i - b_{i-1} - 1|
}

# Tuned for speed (~3× faster than beam=100 / options=10 / distortion=6).
# If quality drops too much, raise beam first; if still slow, set DISTORTION_LIMIT = 0.
DISTORTION_LIMIT = 2
BEAM_SIZE = 32
MAX_OPTIONS_PER_SPAN = 4

# --- Translation options for one source sentence ---------------------------

def build_options(fr_sentence, phrase_table, fallback_lexicon, pw_e_given_f,
                  max_phrase_len, max_options):
  """All (i, j) spans with candidate English phrases.

  For OOV single words we synthesize options from the lexical table (top-k)
  or, failing that, copy the token verbatim. This guarantees every single-word
  span has at least one option, so the future-cost DP never diverges.
  """
  N = len(fr_sentence)
  options = {}
  for i in range(N):
    for j in range(i + 1, min(i + max_phrase_len, N) + 1):
      fr_p = tuple(fr_sentence[i:j])
      if fr_p in phrase_table:
        options[(i, j)] = phrase_table[fr_p][:max_options]
      elif j == i + 1:
        fr_w = fr_p[0]
        alts = pw_e_given_f.get(fr_w)
        candidates = []
        if alts:
          top = sorted(alts.items(), key=lambda x: x[1], reverse=True)[:3]
          for e, p in top:
            if e == NULL_TOKEN:
              continue
            lp = math.log(max(p, 1e-12))
            candidates.append({
                'en': (e,),
                'log_phi_ef': lp,
                'log_phi_fe': lp,
                'log_lex_ef': lp,
                'log_lex_fe': lp,
            })
        if not candidates:
          fb = fallback_lexicon.get(fr_w, fr_w)
          lp = math.log(1e-5)
          candidates.append({
              'en': (fb,),
              'log_phi_ef': lp,
              'log_phi_fe': lp,
              'log_lex_ef': lp,
              'log_lex_fe': lp,
          })
        options[(i, j)] = candidates
  return options

def option_static_score(opt, W):
  """Feature contributions of a phrase pair that don't depend on context:
  everything except LM and distortion."""
  return (W['log_phi_ef']     * opt['log_phi_ef']
        + W['log_phi_fe']     * opt['log_phi_fe']
        + W['log_lex_ef']     * opt['log_lex_ef']
        + W['log_lex_fe']     * opt['log_lex_fe']
        + W['word_bonus']     * len(opt['en'])
        + W['phrase_penalty'])

# --- Future cost (paper §2.2) ----------------------------------------------

def build_future_cost(options, N, lm_word_logprob, W):
  """fc[i][j] = upper-bound score for covering span [i, j), unigram/bigram/
  trigram LM applied as if the span were translated in isolation."""
  NEG_INF = float('-inf')
  fc = [[NEG_INF] * (N + 1) for _ in range(N + 1)]
  for (i, j), opts in options.items():
    best = NEG_INF
    for o in opts:
      lm = 0.0
      ctx = ('<s>', '<s>')
      for w in o['en']:
        lm += lm_word_logprob(w, ctx[0], ctx[1])
        ctx = (ctx[1], w)
      cand = option_static_score(o, W) + W['lm'] * lm
      if cand > best:
        best = cand
    fc[i][j] = best
  for length in range(2, N + 1):
    for i in range(N - length + 1):
      j = i + length
      best = fc[i][j]
      for k in range(i + 1, j):
        combined = fc[i][k] + fc[k][j]
        if combined > best:
          best = combined
      fc[i][j] = best
  return fc

def future_cost_of_mask(mask, N, fc, cache):
  cached = cache.get(mask)
  if cached is not None:
    return cached
  total = 0.0
  i = 0
  while i < N:
    if (mask >> i) & 1:
      i += 1
      continue
    j = i
    while j < N and not ((mask >> j) & 1):
      j += 1
    total += fc[i][j]
    i = j
  cache[mask] = total
  return total

# --- The decoder -----------------------------------------------------------

def phrase_decode(fr_sentence, phrase_table, fallback_lexicon,
                  pw_e_given_f, lm_word_logprob, W,
                  max_phrase_len=PHRASE_MAX_LEN,
                  distortion_limit=DISTORTION_LIMIT,
                  beam_size=BEAM_SIZE,
                  max_options=MAX_OPTIONS_PER_SPAN):
  N = len(fr_sentence)
  if N == 0:
    return []

  options = build_options(
      fr_sentence, phrase_table, fallback_lexicon, pw_e_given_f,
      max_phrase_len, max_options,
  )
  fc = build_future_cost(options, N, lm_word_logprob, W)
  span_mask = {(i, j): ((1 << (j - i)) - 1) << i for (i, j) in options}
  spans = sorted(options.keys())
  full_mask = (1 << N) - 1
  fc_cache = {full_mask: 0.0}

  # Hypothesis: (est, score, mask, last_end, ctx, out_tokens)
  #   est  = score + future_cost(mask)
  #   ctx  = (prev-1, prev) LM bigram context
  # Recombine on key = (mask, last_end, ctx).
  stacks = [dict() for _ in range(N + 1)]
  init_fc = future_cost_of_mask(0, N, fc, fc_cache)
  stacks[0][(0, -1, ('<s>', '<s>'))] = (
      init_fc, 0.0, 0, -1, ('<s>', '<s>'), [])

  for k in range(N):
    if not stacks[k]:
      continue
    hyps = sorted(stacks[k].values(), key=lambda h: h[0], reverse=True)[:beam_size]
    for _, score, mask, last_end, ctx, out in hyps:
      for (i1, i2) in spans:
        m = span_mask[(i1, i2)]
        if mask & m:
          continue
        if last_end >= 0 and distortion_limit >= 0:
          if abs(i1 - (last_end + 1)) > distortion_limit:
            continue
        new_mask = mask | m
        new_count = k + (i2 - i1)
        if last_end >= 0:
          d = abs(i1 - (last_end + 1))
        else:
          d = i1
        dist_cost = W['distortion'] * d
        is_final = (new_count == N)
        for opt in options[(i1, i2)]:
          lm_cost = 0.0
          nctx = ctx
          for w in opt['en']:
            lm_cost += lm_word_logprob(w, nctx[0], nctx[1])
            nctx = (nctx[1], w)
          if is_final:
            lm_cost += lm_word_logprob('</s>', nctx[0], nctx[1])
          new_score = (score
                       + option_static_score(opt, W)
                       + W['lm'] * lm_cost
                       + dist_cost)
          fc_rem = 0.0 if is_final else future_cost_of_mask(new_mask, N, fc, fc_cache)
          est = new_score + fc_rem
          new_last_end = i2 - 1
          key = (new_mask, new_last_end, nctx)
          prev = stacks[new_count].get(key)
          if prev is None or est > prev[0]:
            stacks[new_count][key] = (
                est, new_score, new_mask, new_last_end, nctx, out + list(opt['en']))

  if stacks[N]:
    best = max(stacks[N].values(), key=lambda h: h[1])
    return best[5]
  for k in range(N - 1, -1, -1):
    if stacks[k]:
      return max(stacks[k].values(), key=lambda h: h[1])[5]
  return []

In [13]:
# Paper-style in-domain test: 1,755 held-out Europarl sentences (see training cell).
test_fr_tok = europarl_test_fr_tok
test_en = europarl_test_en_raw   # reference strings for SacreBLEU
test_fr = europarl_test_fr_raw   # for sample display

print(f"Eval set: Europarl held-out (paper §4 style)")
print(f"  Sentences: {len(test_fr_tok):,}")
print(f"  FR length filter: {TEST_SRC_MIN_LEN}-{TEST_SRC_MAX_LEN} tokens")
print(f"Sample FR tokens: {test_fr_tok[0]}")
print(f"Sample EN ref:    {test_en[0][:100]}{'...' if len(test_en[0]) > 100 else ''}")


def parse_sgm(filepath):
  sentences = []
  with open(filepath, encoding='utf-8') as f:
    for line in f:
      line = line.strip()
      if line.startswith('<seg'):
        match = re.search(r'<seg[^>]*>(.*?)</seg>', line)
        if match:
          sentences.append(match.group(1).strip())
  return sentences


# Optional: WMT14 newstest (out-of-domain). Set True to decode + score separately.
RUN_WMT14_EVAL = False
if RUN_WMT14_EVAL:
  fr_test_path = DATA_DIR / 'newstest2014-fren-src.fr.sgm'
  en_test_path = DATA_DIR / 'newstest2014-fren-ref.en.sgm'
  wmt_fr = parse_sgm(fr_test_path)
  wmt_en = parse_sgm(en_test_path)
  wmt_fr_tok = [tokenize(s) for s in wmt_fr]
  print(f"WMT14 newstest loaded: {len(wmt_fr_tok):,} sentences (not decoded unless you add a loop).")

Eval set: Europarl held-out (paper §4 style)
  Sentences: 1,755
  FR length filter: 5-15 tokens
Sample FR tokens: ['beaucoup', 'de', 'ces', 'enfants', 'qui', 'ne', 'vont', 'pas', 'à', "l'", 'école', 'travaillent.']
Sample EN ref:    many of these children who are not in school are working.


In [14]:
# Evaluate on Europarl held-out test (1,755 sents, FR len 5–15) — Koehn et al. 2003 §4.
import time

t0 = time.time()
phrase_translations = []
for idx, fr_tokens in enumerate(test_fr_tok):
  pred = phrase_decode(
      fr_tokens,
      phrase_table,
      fallback_lexicon,
      pw_e_given_f,
      lm_word_logprob,
      WEIGHTS,
      max_phrase_len=PHRASE_MAX_LEN,
      distortion_limit=DISTORTION_LIMIT,
      beam_size=BEAM_SIZE,
      max_options=MAX_OPTIONS_PER_SPAN,
  )
  phrase_translations.append(detokenize(pred))
  if (idx + 1) % 100 == 0:
    elapsed = time.time() - t0
    rate = (idx + 1) / elapsed
    eta = (len(test_fr_tok) - (idx + 1)) / max(rate, 1e-6)
    print(f'Decode {idx + 1}/{len(test_fr_tok)}  '
          f'({rate:.1f} sent/s, eta {eta/60:.1f} min)')

elapsed = time.time() - t0
print(f'\nTotal decode time: {elapsed/60:.1f} min ({len(test_fr_tok)/elapsed:.2f} sent/s)')

# lowercase=True is crucial: our predictions are lowercase, references aren't.
phrase_bleu = sacrebleu.corpus_bleu(phrase_translations, [test_en], lowercase=True)
print(f'\nEuroparl held-out BLEU (lowercase): {phrase_bleu.score:.2f}')

cased_bleu = sacrebleu.corpus_bleu(phrase_translations, [test_en])
print(f'Europarl held-out BLEU (cased):     {cased_bleu.score:.2f}')

print('\nSamples:')
for i in range(5):
  print(f'FR : {test_fr[i]}')
  print(f'REF: {test_en[i]}')
  print(f'PB : {phrase_translations[i]}')
  print()

Decode 100/1755  (92.8 sent/s, eta 0.3 min)
Decode 200/1755  (91.8 sent/s, eta 0.3 min)
Decode 300/1755  (92.6 sent/s, eta 0.3 min)
Decode 400/1755  (94.1 sent/s, eta 0.2 min)
Decode 500/1755  (88.0 sent/s, eta 0.2 min)
Decode 600/1755  (88.5 sent/s, eta 0.2 min)
Decode 700/1755  (88.3 sent/s, eta 0.2 min)
Decode 800/1755  (85.2 sent/s, eta 0.2 min)
Decode 900/1755  (85.6 sent/s, eta 0.2 min)
Decode 1000/1755  (82.9 sent/s, eta 0.2 min)
Decode 1100/1755  (83.8 sent/s, eta 0.1 min)
Decode 1200/1755  (84.5 sent/s, eta 0.1 min)
Decode 1300/1755  (82.9 sent/s, eta 0.1 min)
Decode 1400/1755  (83.6 sent/s, eta 0.1 min)
Decode 1500/1755  (84.2 sent/s, eta 0.1 min)
Decode 1600/1755  (84.9 sent/s, eta 0.0 min)
Decode 1700/1755  (85.2 sent/s, eta 0.0 min)

Total decode time: 0.3 min (84.85 sent/s)

Europarl held-out BLEU (lowercase): 23.79
Europarl held-out BLEU (cased):     23.79

Samples:
FR : beaucoup de ces enfants qui ne vont pas à l' école travaillent.
REF: many of these children who are n